---

# ~ DAC Find IT! 2026: Advanced Face Anti-Spoofing ~
*6-Class Face Liveness Detection*

---

## Table of Contents

1. [**Data Preprocessing & Advanced Augmentation**](#1-data-preprocessing--advanced-augmentation)
2. [**Building Swin and CNN Architectures**](#2-building-swin-and-cnn-architectures)
3. [**Training Strategy (Warm-up & Loss Functions)**](#3-training-strategy-warm-up--loss-functions)
4. [**Probability Calibration (Thresholding)**](#4-probability-calibration-thresholding)
5. [**TTA Inference & Submission Generation**](#5-tta-inference--submission-generation)

---

## Libraries

In [ ]:
!pip install -q timm albumentations opencv-python-headless tqdm scikit-learn

In [ ]:
import os
import gc
import warnings
import random

import numpy as np
import pandas as pd
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from scipy.optimize import minimize
from tqdm.auto import tqdm

# --- Environment & Memory Configuration ---
warnings.filterwarnings("ignore")
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

def seed_everything(seed=42):
    """
    Ensures reproducibility across numpy, random, and pytorch.
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

---

# Data Preprocessing & Advanced Augmentation <a name="1"></a>

---

**Technical Challenge & Strategy**
The primary challenge in face spoofing detection (e.g., silicone masks, digital screens, or high-quality prints) is the model's tendency to overfit to background noise or camera-specific characteristics rather than learning genuine liveness features.

 To achieve peak performance and robust generalization, this notebook adopts two foundational strategies:
 1. **Stratified K-Fold Cross-Validation:** Maintains a consistent 6-class distribution across all experimental folds.
 2. **Advanced Data Augmentation:** Simulates compression artifacts, extreme lighting conditions, and sensor noise (ISONoise/GaussNoise) to force the model to capture microscopic texture patterns instead of relying on image resolution.

## Import & Global Configuration 

In [ ]:
class CFG:
    """Global hyperparameter configuration"""
    seed = 42
    img_size = 384
    batch_size = 8
    n_folds = 5
    num_classes = 6
    num_workers = 2
    
    # Paths configuration
    base_path = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026"
    train_dir = os.path.join(base_path, "train")
    
    # Compute environment
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Dataframe creation & stratified K-FOLD

In [ ]:
# Directory-to-Label Mapping
class_names = sorted(os.listdir(CFG.train_dir))
label2id = {name: i for i, name in enumerate(class_names)}
id2label = {i: name for name, i in label2id.items()}

data = []
for class_name in class_names:
    class_dir = os.path.join(CFG.train_dir, class_name)
    if os.path.isdir(class_dir):
        for img_name in os.listdir(class_dir):
            data.append({
                "image_path": os.path.join(class_dir, img_name),
                "label": label2id[class_name]
            })

df_all = pd.DataFrame(data)

# Stratified K-Fold Split
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
df_all['fold'] = -1

for fold, (train_idx, valid_idx) in enumerate(skf.split(X=df_all, y=df_all['label'])):
    df_all.loc[valid_idx, 'fold'] = fold

# Distribution Summary
print(f"Total Samples: {len(df_all)}")
display(df_all.groupby(['fold', 'label']).size().unstack(fill_value=0))

## Pipeline Augmentasi 

In [ ]:
# Training Augmentations: Optimized for Anti-Spoofing challenges
train_transforms = A.Compose([
    A.RandomResizedCrop(size=(CFG.img_size, CFG.img_size), scale=(0.8, 1.0), p=1.0),
    A.HorizontalFlip(p=0.5),
    
    # Block 1: Digital camera artifacts & sensor noise simulation
    A.OneOf([
        A.ImageCompression(p=1.0), 
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),                        
        A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0), 
        A.GaussNoise(p=1.0),                     
    ], p=0.5),
    
    # Block 2: Robustness against lighting variations (screens/paper prints)
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
    
    # Block 3: Spatial distortion & Occlusion (In-the-wild simulation)
    A.Affine(scale=(0.95, 1.05), translate_percent=(-0.05, 0.05), rotate=(-15, 15), p=0.5),
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 32), hole_width_range=(8, 32), fill=0, p=0.5),
    
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Validation Transforms: Standard evaluation preprocessing
valid_transforms = A.Compose([
    A.Resize(height=CFG.img_size, width=CFG.img_size),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

## Kelas pytorch dataset

In [ ]:
class SpoofingDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        image = cv2.imread(img_path)
        
        # Error handling for corrupt images or invalid paths
        if image is None:
            image = np.zeros((CFG.img_size, CFG.img_size, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            augmented = self.transforms(image=image)
            image = augmented['image']

        label = self.df.loc[idx, 'label']
        return image, torch.tensor(label, dtype=torch.long)

---

# Building Swin and CNN Architectures <a name="2"></a>

---

### Face Anti-Spoofing (FAS) Architecture

In the Face Anti-Spoofing (FAS) challenge, a model cannot rely on a single perspective. A 3D silicone mask may possess textures nearly identical to real skin (potentially deceiving a CNN), while a high-resolution photo on a mobile screen might maintain perfect facial proportions (potentially deceiving a Transformer). 

To address this, we implemented a **Heterogeneous Architecture** consisting of two complementary models:

---

#### 1. Swin Transformer (`swin_base_patch4`)
The Swin Transformer operates by partitioning the image into patches and computing relationships between these areas using **Shifted Window Attention**. This model acts as the **"Macro-Perspective Eye."**



* **Primary Function:** Detects geometric anomalies, global lighting inconsistencies, and unnatural facial proportions commonly found in mannequins or rigid masks.
* **Custom Modifications:** We replaced the default classification head with a **Custom Head (Linear → BatchNorm → ReLU → Dropout)**. `ReLU` provides sharper non-linear decision boundaries compared to default activations, making it ideal for distinguishing subtle spoofing features.
* **Optimization:** We enabled `Gradient Checkpointing` to facilitate training with high-resolution images ($384 \times 384$) without encountering VRAM **Out of Memory (OOM)** errors.

#### 2. EfficientNet V2 (`tf_efficientnetv2_m`)
EfficientNet is a pure Convolutional Neural Network (CNN) architecture that utilizes sliding filters. This model acts as the **"Digital Magnifying Glass."**

* **Primary Function:** Highly sensitive to pixel-level texture anomalies through its **Local Receptive Field**. 
* **Spoof Detection:** It is exceptionally reliable at detecting **Moiré patterns** (the aliasing interference caused when photographing digital screens) or the micro-textures of paper grain in printed spoof attacks.



---

### Why the Hybrid Approach?
By combining these two architectures, the pipeline benefits from **Global Semantic Awareness** (Transformer) and **Local Texture Sensitivity** (CNN), creating a robust defense against both physical and digital spoofing attempts.

## Model Architectures 

In [ ]:
class SwinWithCustomHead(nn.Module):
    """Swin Transformer with a ReLU-based Custom Head for 6-class classification."""
    def __init__(self, model_name, num_classes=6):
        super(SwinWithCustomHead, self).__init__()
        
        # Load backbone without the default classification head
        self.backbone = timm.create_model(
            model_name, 
            pretrained=True, 
            num_classes=0,       
            drop_rate=0.3,       
            attn_drop_rate=0.2   
        )
        
        # VRAM Optimization: Gradient Checkpointing
        self.backbone.set_grad_checkpointing(enable=True)
        
        num_features = self.backbone.num_features
        
        # Custom head implementation using ReLU activation
        self.custom_head = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        output = self.custom_head(features)
        return output

def create_swin_model():
    """Initializes Swin Transformer Base (384ms resolution)."""
    model_name = 'swin_base_patch4_window12_384.ms_in22k'
    return SwinWithCustomHead(model_name, num_classes=6).to(CFG.device)

def create_cnn_model():
    """Initializes EfficientNetV2 Medium."""
    model_name = 'tf_efficientnetv2_m.in21k_ft_in1k'
    model = timm.create_model(
        model_name, 
        pretrained=True, 
        num_classes=6,
        drop_rate=0.3,       
        drop_path_rate=0.2   
    )
    return model.to(CFG.device)

## Model Varification

In [ ]:
def verify_architectures():
  
    
    # Dummy input (Batch=2, Channel=3, Size=384x384)
    dummy_input = torch.randn(2, 3, 384, 384).to(CFG.device)
    
    # Inisialisasi model secara temporer
    swin_temp = create_swin_model()
    cnn_temp = create_cnn_model()
    
    with torch.no_grad():
        out_swin = swin_temp(dummy_input)
        out_cnn = cnn_temp(dummy_input)
    
    print(f" Swin Output Shape : {out_swin.shape}")
    print(f" CNN Output Shape  : {out_cnn.shape}")
    
    # Bersihkan memori GPU secara total
    del swin_temp, cnn_temp, dummy_input
    torch.cuda.empty_cache()
    gc.collect()

verify_architectures()

---

# Training Strategy (Warm-up & Loss Functions) <a name="3"></a>

---

Training a Face Anti-Spoofing (FAS) model is not just about minimizing loss as quickly as possible. These models are highly susceptible to **Overconfidence** and **Class Imbalance** regarding difficult-to-recognize features (hard examples). To achieve robust performance, we implement three primary strategies:

1. **Focal Loss + Label Smoothing**
   Instead of standard Cross-Entropy, we utilize **Focal Loss** to compel the model to focus on learning from samples that are frequently misclassified (e.g., distinguishing real skin from high-resolution paper prints). Additionally, **Label Smoothing (0.1)** is applied to prevent the model from assigning absolute (100%) probabilities. This has been proven to make the **decision boundary** more tolerant and improve overall generalization.

2. **Cosine Annealing Learning Rate**
   We employ a scheduler that decays the **Learning Rate (LR)** following a cosine curve. This helps the model converge into broader and more stable local minima, which significantly enhances validation scores during the final stages of training.

3. **Automatic Mixed Precision (AMP)**
   Training speed is accelerated by up to 2x and VRAM usage is optimized using `torch.amp`. This allows us to train high-capacity models on Kaggle GPUs without encountering **Out of Memory (OOM)** errors.

## Custom Loss Function

In [ ]:
class FocalLossWithSmoothing(nn.Module):
    """
    Hybrid loss function combining Focal Loss (focusing on hard samples) 
    and Label Smoothing (preventing model overconfidence).
    """
    def __init__(self, alpha=1.0, gamma=2.0, smoothing=0.1, reduction='mean'):
        super(FocalLossWithSmoothing, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing
        self.reduction = reduction

    def forward(self, inputs, targets):
        num_classes = inputs.size(1)
        
        # 1. Apply Label Smoothing
        smooth_target = F.one_hot(targets, num_classes=num_classes).float()
        smooth_target = smooth_target * (1.0 - self.smoothing) + (self.smoothing / num_classes)
        
        # 2. Calculate Focal Loss components
        log_pt = F.log_softmax(inputs, dim=1)
        pt = torch.exp(log_pt)
        
        ce_loss = -(smooth_target * log_pt).sum(dim=1)
        focal_term = self.alpha * (1 - pt.gather(1, targets.view(-1, 1)).squeeze(1)) ** self.gamma
        
        loss = focal_term * ce_loss
        
        if self.reduction == 'mean':
            return loss.mean()
        return loss.sum()

## Training Loop Pelatihan 10-Model Ensembel

In [ ]:
def train_fold(fold, train_loader, valid_loader, model_func, model_prefix, epochs=20, lr=1e-5):
    print(f"\n{'='*40}\n TRAINING: {model_prefix.upper()} | FOLD {fold}\n{'='*40}")
    
    model = model_func().to(CFG.device)
    criterion = FocalLossWithSmoothing(gamma=2.0, smoothing=0.1) 
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-7)
    scaler = torch.amp.GradScaler('cuda') # AMP for memory efficiency
    
    best_val_f1 = 0.0
    patience, patience_counter = 5, 0
    save_path = f"best_{model_prefix}_fold_{fold}.pth"

    for epoch in range(epochs):
        # 1. Training Phase
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}/{epochs} [TRAIN]", leave=False)
        
        for images, labels in pbar:
            images, labels = images.to(CFG.device), labels.to(CFG.device)
            optimizer.zero_grad()
            
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Gradient clipping for stability
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item() * images.size(0)
            
        avg_train_loss = train_loss / len(train_loader.dataset)
        
        # 2. Validation Phase
        model.eval()
        val_loss, all_preds, all_labels = 0.0, [], []
        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(CFG.device), labels.to(CFG.device)
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                    
                val_loss += loss.item() * images.size(0)
                all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        avg_val_loss = val_loss / len(valid_loader.dataset)
        val_macro_f1 = f1_score(all_labels, all_preds, average='macro')
        scheduler.step()
        
        print(f"Ep {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val F1: {val_macro_f1:.4f}")
        
        # 3. Model Checkpointing & Early Stopping
        if val_macro_f1 > best_val_f1:
            best_val_f1 = val_macro_f1
            torch.save(model.state_dict(), save_path)
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= patience:
            print(f"Early stopping triggered: Model convergence reached.")
            break
            
    # Memory Management: Clear cache for the next model
    del model, optimizer, scaler, criterion
    torch.cuda.empty_cache(); gc.collect()
    return best_val_f1

# --- MULTI-MODEL ENSEMBLE EXECUTION ---
scores = {"swin": [], "cnn": []}

print(f"\n Starting 10-Model Ensemble Training (5-Fold CV)...")

for fold in range(CFG.n_folds):
    print(f"\n\n{'='*50}\n PREPARING DATA FOR FOLD {fold}\n{'='*50}")
    
    # Subset data for current fold
    train_ds = SpoofingDataset(df_all[df_all['fold'] != fold], train_transforms)
    valid_ds = SpoofingDataset(df_all[df_all['fold'] == fold], valid_transforms)
    
    train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, drop_last=True, num_workers=2)
    valid_loader = DataLoader(valid_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=2)
    
    # Training Pipeline: Swin Transformer followed by EfficientNetV2
    for prefix, m_func in [("swin", create_swin_model), ("cnn", create_cnn_model)]:
        best_f1 = train_fold(
            fold=fold,
            train_loader=train_loader,
            valid_loader=valid_loader,
            model_func=m_func,
            model_prefix=prefix,
            epochs=20, 
            lr=1e-5 if prefix == "swin" else 2e-5 # CNN stability: slightly higher LR
        )
        scores[prefix].append(best_f1)

# --- FINAL PERFORMANCE RECAPITULATION ---
print(f"\n{'='*50}\n FINAL ENSEMBLE RECAPITULATION\n{'='*50}")
for model_type, model_scores in scores.items():
    print(f"--- {model_type.upper()} ---")
    for i, s in enumerate(model_scores): 
        print(f"Fold {i} F1: {s:.4f}")
    print(f"Average {model_type.upper()}: {np.mean(model_scores):.4f}\n")

## Report 

In [ ]:
print(f"\n{'='*50}\n10-MODEL ENSEMBLE RECAPITULATION\n{'='*50}")
for model_type, model_scores in scores.items():
    print(f"--- {model_type.upper()} ---")
    for i, s in enumerate(model_scores): 
        print(f"Fold {i} F1-Score: {s:.4f}")
    print(f"Average {model_type.upper()} F1: {np.mean(model_scores):.4f}\n")

---

# Probability Calibration (Thresholding) <a name="4"></a>

---

Deep Learning models often encounter **Probability Calibration** issues. For instance, a model might be overly conservative in assigning high probabilities to minority classes such as `fake_unknown`. Consequently, these classes are consistently outranked during the `argmax` operation, even when the model's internal features correctly identify them.

This stage is designed to rectify these systematic biases:

1. **OOF (Out-of-Fold) Prediction Extraction**
   We aggregate the softmax predictions from the 5-fold cross-validation using their respective validation sets. Because this data was unseen during the training phase, it serves as a reliable proxy for how the model will generalize to the actual test set.

2. **Class-Weight Calibration (Powell Method)**
   We employ a **SciPy optimization algorithm (Powell's Method)** to determine six optimal multipliers—one for each class. The algorithm iteratively searches for a weight combination that yields the **highest Macro F1-Score** on the OOF data. By scaling the raw probabilities, we balance the model's sensitivity across all classes, ensuring that minority classes are not suppressed by majority class bias.

## OOF Inference Engine 

In [ ]:
def extract_oof_probs(model_func, path_template, data_df):
    """
    Extracts Out-of-Fold (OOF) probability predictions from trained models.
    
    Args:
        model_func: Function to initialize the model architecture.
        path_template: String template for the saved model weights path.
        data_df: DataFrame containing the 'fold' assignments.
        
    Returns:
        numpy.ndarray: Array of OOF probability predictions.
    """
    oof_probs = np.zeros((len(data_df), CFG.num_classes))
    
    for fold in range(CFG.n_folds):
        # Identify validation indices for the current fold
        val_idx = data_df[data_df['fold'] == fold].index
        valid_loader = DataLoader(
            SpoofingDataset(data_df.iloc[val_idx], transforms=valid_transforms), 
            batch_size=CFG.batch_size, 
            shuffle=False, 
            num_workers=CFG.num_workers
        )
        
        # Load the best weights for this fold
        model = model_func().to(CFG.device)
        model.load_state_dict(torch.load(path_template.format(fold), map_location=CFG.device))
        model.eval()
        
        fold_preds = []
        with torch.no_grad():
            for images, _ in valid_loader:
                images = images.to(CFG.device)
                # Compute softmax probabilities
                probs = F.softmax(model(images), dim=1)
                fold_preds.append(probs.cpu().numpy())
        
        oof_probs[val_idx] = np.vstack(fold_preds)
        
        # Memory Management: Clear VRAM for the next fold
        del model
        torch.cuda.empty_cache()
        gc.collect()
        
    return oof_probs

## Execution: oof extraction 

In [ ]:
# Ekstraksi probabilitas untuk kedua arsitektur
oof_probs_swin = extract_oof_probs(create_swin_model, "best_swin_fold_{}.pth", df_all)
oof_probs_cnn  = extract_oof_probs(create_cnn_model, "best_cnn_fold_{}.pth", df_all)
oof_labels     = df_all['label'].values

# Ensemble Awal: 60% Swin, 40% CNN
oof_probs_ensemble = (oof_probs_swin * 0.60) + (oof_probs_cnn * 0.40)

## Optimization & Calibration 

In [ ]:
# Initial Evaluation: Base ensemble performance
preds_before = np.argmax(oof_probs_ensemble, axis=1)
print("--- ENSEMBLE PERFORMANCE (PRE-CALIBRATION) ---")
print(classification_report(oof_labels, preds_before, target_names=class_names, digits=4))

# Optimization Objective: Maximize Macro F1-Score by scaling class probabilities
def objective(weights):
    """
    Applies weights to probabilities and calculates negative F1-Score for minimization.
    """
    calibrated = oof_probs_ensemble * weights
    preds = np.argmax(calibrated, axis=1)
    return -1.0 * f1_score(oof_labels, preds, average='macro')

# Run Optimizer: Using Powell's method to find optimal class-specific weights
print("\nStarting search for optimal class-weights...")
res = minimize(
    objective, 
    x0=[1.0] * CFG.num_classes, 
    method='Powell', 
    bounds=[(0.5, 2.0)] * CFG.num_classes
)

# Apply Optimized Weights
best_weights = res.x
final_probs = oof_probs_ensemble * best_weights
preds_final = np.argmax(final_probs, axis=1)

# Post-Calibration Summary
print("\n--- OPTIMIZED PERFORMANCE (POST-CALIBRATION) ---")
print(classification_report(oof_labels, preds_final, target_names=class_names, digits=4))
print(f"\nOptimal Class-Weights for Inference:\n{list(np.round(best_weights, 4))}")

---

# TTA Inference & Submission Generation <a name="4"></a>

---

The final stage of the pipeline involves generating predictions for the competition dataset. To achieve a competitive score, we implement a robust inference strategy utilizing three key techniques:

1. **Test-Time Augmentation (TTA)**
   We perform inference twice for every image—once on the original image and once on its horizontal flip. Averaging these two predictions helps the model remain invariant to facial orientation and significantly reduces variance in the output.

2. **Weighted 10-Model Ensemble**
   We aggregate the probabilities from our **10 distinct models** (5 Swin Transformer folds and 5 EfficientNetV2 folds). A weighted average is applied (e.g., 85% Swin and 15% CNN) to leverage the Swin Transformer's superior global context while still benefiting from the CNN's sensitivity to local texture anomalies.

3. **Post-Calibration (Scaling)**
   The final ensemble probabilities are scaled using the optimal class-specific weights derived from the **Powell Optimization** (Stage 4). This step is crucial for maximizing the **Macro F1-Score** on the leaderboard by ensuring that the decision thresholds are perfectly tuned across all 6 classes.

## Inference Dataset & TTA preaparation 

In [ ]:
# Load Submission Template
test_dir = os.path.join(CFG.base_path, "test")
sub_df = pd.read_csv(os.path.join(CFG.base_path, "samplesubmission.csv"))

class TestDatasetTTA(Dataset):
    """
    Dataset class for inference with Test-Time Augmentation (TTA).
    Generates both original and horizontally flipped versions of each image.
    """
    def __init__(self, df, test_dir):
        self.df = df
        self.test_dir = test_dir
        self.transforms = A.Compose([
            A.Resize(height=CFG.img_size, width=CFG.img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])
        
        # File mapping to handle potential extension mismatches
        self.existing_files = {
            os.path.splitext(f)[0]: f 
            for f in os.listdir(test_dir) 
            if os.path.isfile(os.path.join(test_dir, f))
        }

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx]['id'])
        img_id_clean = os.path.splitext(img_id)[0] 
        
        # Safe Image Loading: Handling missing or corrupt files
        if img_id_clean in self.existing_files:
            img_path = os.path.join(self.test_dir, self.existing_files[img_id_clean])
            image = cv2.imread(img_path) 
            if image is not None:
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            else:
                image = np.zeros((CFG.img_size, CFG.img_size, 3), dtype=np.uint8)
        else:
            image = np.zeros((CFG.img_size, CFG.img_size, 3), dtype=np.uint8)
        
        # Apply Transforms for TTA: Original & Flipped
        img_orig = self.transforms(image=image)['image']
        img_flip = self.transforms(image=cv2.flip(image, 1))['image']

        return img_id, img_orig, img_flip

# Initialize Test DataLoader
test_loader = DataLoader(
    TestDatasetTTA(sub_df, test_dir), 
    batch_size=CFG.batch_size, 
    shuffle=False, 
    num_workers=CFG.num_workers
)

## OOF inference Engine & Ensemble 

In [ ]:
def run_tta_inference(model_func, model_prefix, weight_ratio):
    """
    Executes 5-Fold TTA inference for a specific model architecture.
    
    Args:
        model_func: Function to initialize the model architecture.
        model_prefix: String prefix identifying the model type ('swin' or 'cnn').
        weight_ratio: Scalar to weight the model's contribution to the final ensemble.
        
    Returns:
        numpy.ndarray: Weighted ensemble probability predictions.
    """
    print(f"\nExecuting {model_prefix.upper()} 5-Fold TTA Inference (Weight: {weight_ratio*100:.0f}%)")
    
    ensemble_probs = np.zeros((len(sub_df), CFG.num_classes))
    
    for fold in range(CFG.n_folds):
        # Load fold-specific model weights
        model = model_func().to(CFG.device)
        model.load_state_dict(torch.load(f'best_{model_prefix}_fold_{fold}.pth', map_location=CFG.device))
        model.eval()

        fold_probs = []
        with torch.no_grad():
            for _, imgs_orig, imgs_flip in tqdm(test_loader, desc=f"Fold {fold}"):
                # TTA Averaging: Mean of original and horizontal flip predictions
                p_orig = F.softmax(model(imgs_orig.to(CFG.device)), dim=1)
                p_flip = F.softmax(model(imgs_flip.to(CFG.device)), dim=1)
                fold_probs.append(((p_orig + p_flip) / 2.0).cpu().numpy())

        # Aggregate K-Fold predictions
        ensemble_probs += np.vstack(fold_probs) / CFG.n_folds
        
        # Memory Management: Clear VRAM for the next fold
        del model
        torch.cuda.empty_cache()
        gc.collect()
        
    return ensemble_probs * weight_ratio

# --- Modular Inference Execution ---
# Note: Ratios are optimized based on OOF performance (85% Swin, 15% CNN)
swin_weighted_probs = run_tta_inference(create_swin_model, "swin", weight_ratio=0.85)
cnn_weighted_probs  = run_tta_inference(create_cnn_model, "cnn", weight_ratio=0.15)

# 10-Model Hybrid Ensemble Generation
final_ensemble_probs = swin_weighted_probs + cnn_weighted_probs

# --- Post-Processing Calibration ---
# Apply the optimal class weights obtained from Stage 4 (Powell Optimization)
# Replace with your actual optimized weight array for final submission
best_powell_weights = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0]) 
calibrated_probs = final_ensemble_probs * best_powell_weights

## Submission Generation 

In [ ]:
final_preds = np.argmax(calibrated_probs, axis=1)

# Mapping Label
class_names = ['fake_mannequin', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_unknown', 'realperson']
id2label = {i: name for i, name in enumerate(class_names)}

sub_df['label'] = [id2label[p] for p in final_preds]
sub_df.to_csv('submission.csv', index=False)